<a href="https://colab.research.google.com/github/ReginaGH/Enter_to_NC/blob/main/rnn_balakirev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Описание
Строим протсую реккурентную нейронную сеть, на вход которой будем подавать отдельные символы, а на выходе она будет строить прогноз следующего символа. В качестве обучения у нас будет текст

## Libraries

In [1]:
import numpy as np
import re

import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense, SimpleRNN, Input
from keras.models import Sequential
from keras.preprocessing.text import Tokenizer

In [19]:
with open('/content/drive/MyDrive/text_selfedu.txt', 'r', encoding='utf-8') as f:
  text = f.read()
  text = text.replace('\ufeff', '')  # убираем первый невидимый символ
  text = re.sub(r'[^А-я]', ' ', text)  # оставляем только символы русского алфавита
  # заменяем все символы кроме кириллицы на пустые символы

Tokenizer params:
* num_words - максимальное кол-во слов/символов, которое вернёт токенайзер (если элементов будет больше, то останутся наиболее повторяющиеся в тексте)
* filters - исключаемые из текста символы
* lower = True перевод символов в нижний регистр
* split - разделение слов по символу пробела
* char_level=False если False, то текст делится на слова , иначе на символы

In [20]:
# @ title Парсим текст как последовательность символов
num_characters = 34  # 33 буквы + пробел
tokenizer = Tokenizer(num_words=num_characters, char_level=True)  # разложение текста на составляющие элементы (через создание экземпляра класса)
tokenizer.fit_on_texts(text)
print(tokenizer.word_index)  # можно посмотреть разбивку символов по индексам

{' ': 1, 'о': 2, 'е': 3, 'т': 4, 'и': 5, 'н': 6, 'а': 7, 'с': 8, 'в': 9, 'л': 10, 'р': 11, 'м': 12, 'д': 13, 'ь': 14, 'п': 15, 'у': 16, 'к': 17, 'ы': 18, 'я': 19, 'б': 20, 'з': 21, 'ч': 22, 'г': 23, 'й': 24, 'ж': 25, 'ш': 26, 'х': 27, 'ю': 28, 'э': 29, 'щ': 30, 'ц': 31, 'ф': 32, 'ъ': 33}


text_to_matrix  -разбивает текст на вектора, где каждый вектор это кодировка буквы

In [21]:
inp_chars = 6
data = tokenizer.texts_to_matrix(text)
n = data.shape[0] - inp_chars

In [22]:
X = np.array([data[i:i + inp_chars, :] for i in range(n)])
Y = data[inp_chars:]
print(data.shape)

(10701, 34)


## Create a model NN

In [28]:
model = Sequential()
model.add(Input((inp_chars, num_characters)))
model.add(SimpleRNN(256, activation='tanh'))
model.add(Dense(num_characters, activation='softmax'))
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 simple_rnn_2 (SimpleRNN)    (None, 256)               74496     
                                                                 
 dense_2 (Dense)             (None, 34)                8738      
                                                                 
Total params: 83234 (325.13 KB)
Trainable params: 83234 (325.13 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [29]:
model.compile(
    loss='categorical_crossentropy',
    metrics=['accuracy'],
    optimizer='adam'
)

In [30]:
history = model.fit(X, Y, batch_size=32, epochs=100)
# 256 [==============================] - 7s 20ms/step - loss: 0.4177 - accuracy: 0.8690

Epoch 1/100
335/335 [==============================] - 5s 11ms/step - loss: 2.7931 - accuracy: 0.2299
Epoch 2/100
335/335 [==============================] - 3s 10ms/step - loss: 2.4595 - accuracy: 0.3100
Epoch 3/100
335/335 [==============================] - 3s 8ms/step - loss: 2.3149 - accuracy: 0.3340
Epoch 4/100
335/335 [==============================] - 3s 8ms/step - loss: 2.2489 - accuracy: 0.3476
Epoch 5/100
335/335 [==============================] - 3s 8ms/step - loss: 2.2118 - accuracy: 0.3472
Epoch 6/100
335/335 [==============================] - 5s 13ms/step - loss: 2.1780 - accuracy: 0.3655
Epoch 7/100
335/335 [==============================] - 3s 8ms/step - loss: 2.1684 - accuracy: 0.3585
Epoch 8/100
335/335 [==============================] - 3s 8ms/step - loss: 2.1419 - accuracy: 0.3575
Epoch 9/100
335/335 [==============================] - 3s 8ms/step - loss: 2.1204 - accuracy: 0.3679
Epoch 10/100
335/335 [==============================] - 4s 10ms/step - loss: 2.1040 - ac

In [31]:
def build_phrase(inp_str, str_len=50):
  for i in range(str_len):
    x = []
    for j in range(i, i + inp_chars):
      x.append(tokenizer.texts_to_matrix(inp_str[j]))  # преобразуем символы в one-hot-encoding

    x = np.array(x)
    inp = x.reshape(1, inp_chars, num_characters)

    pred = model.predict(inp)  # получаем вектор из 34 элементов
    d = tokenizer.index_word[pred.argmax(axis=1)[0]]

    inp_str += d

  return inp_str

res = build_phrase('утренн')

print(res)

1/1 [==============================] - 0s 21ms/step
утренним проблемам помешать великими и взять что толку о
